# Native Video Embedding — Qwen3-VL

Tests native video input via the transformers API directly (no SentenceTransformers).

**Approach:** Pass video file directly using `"type": "video"` — the model handles frame extraction internally via `qwen_vl_utils`.

**Compare with:** Our existing frame-extraction approach (OpenCV → PIL → embed_images).

Runtime: GPU (T4 or better recommended).

In [ ]:
!pip install -q "transformers>=4.57.0" qwen-vl-utils torch bitsandbytes "chromadb==0.5.23" Pillow

## Configuration

In [ ]:
# Model settings
MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"  # or Qwen/Qwen3-VL-Embedding-8B
USE_4BIT = False       # set True for 8B on T4
TARGET_DIMS = 768      # MRL truncation — use fewer dims for faster search (max: 2048 for 2B, 4096 for 8B)
VIDEO_FPS = 1.0        # frames per second to sample from video
VIDEO_MAX_FRAMES = 32  # cap on frames passed to model

## Build embedder

In [ ]:
import os, time, sys, importlib.util
import torch
import torch.nn.functional as F
from transformers.models.qwen3_vl.processing_qwen3_vl import Qwen3VLProcessor
from qwen_vl_utils import process_vision_info

# Write the model class to a real .py file so transformers can inspect it
# (Jupyter __main__ has no __file__, which causes AttributeError)
_class_code = """
from transformers.models.qwen3_vl.modeling_qwen3_vl import (
    Qwen3VLPreTrainedModel, Qwen3VLModel, Qwen3VLConfig,
)

class _Qwen3VLForEmbedding(Qwen3VLPreTrainedModel):
    config: Qwen3VLConfig
    def __init__(self, config):
        super().__init__(config)
        self.model = Qwen3VLModel(config)
        self.post_init()
    def get_input_embeddings(self):
        return self.model.get_input_embeddings()
    def set_input_embeddings(self, value):
        self.model.set_input_embeddings(value)
    def forward(self, input_ids=None, attention_mask=None, position_ids=None,
                past_key_values=None, inputs_embeds=None, pixel_values=None,
                pixel_values_videos=None, image_grid_thw=None,
                video_grid_thw=None, cache_position=None, **kwargs):
        return self.model(
            input_ids=input_ids, pixel_values=pixel_values,
            pixel_values_videos=pixel_values_videos,
            image_grid_thw=image_grid_thw, video_grid_thw=video_grid_thw,
            position_ids=position_ids, attention_mask=attention_mask,
            past_key_values=past_key_values, inputs_embeds=inputs_embeds,
            cache_position=cache_position, **kwargs,
        )
"""

_module_path = "/tmp/_qwen3vl_for_embedding.py"
with open(_module_path, "w") as _f:
    _f.write(_class_code)

_spec = importlib.util.spec_from_file_location("_qwen3vl_for_embedding", _module_path)
_mod = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _mod  # register so transformers can find it via sys.modules
_spec.loader.exec_module(_mod)
_Qwen3VLForEmbedding = _mod._Qwen3VLForEmbedding


class Qwen3VLEmbedder:
    def __init__(self, model_name=MODEL_NAME, dimensions=TARGET_DIMS, use_4bit=USE_4BIT):
        self.model_name = model_name
        self.dimensions = dimensions
        self.use_4bit = use_4bit
        self._model = None
        self._processor = None

    def _load(self):
        if self._model is not None:
            return

        if torch.cuda.is_available():
            device, dtype = "cuda", torch.bfloat16
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device, dtype = "mps", torch.float16
        else:
            device, dtype = "cpu", torch.float32

        load_kwargs = {"trust_remote_code": True}
        if self.use_4bit and device == "cuda":
            from transformers import BitsAndBytesConfig
            load_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            print("Using 4-bit quantization")
        else:
            load_kwargs["torch_dtype"] = dtype

        print(f"Loading {self.model_name} on {device}...")
        self._processor = Qwen3VLProcessor.from_pretrained(self.model_name, padding_side="right")
        self._model = _Qwen3VLForEmbedding.from_pretrained(self.model_name, **load_kwargs)
        if "quantization_config" not in load_kwargs:
            self._model = self._model.to(device)
        self._model.eval()
        print(f"✓ Model ready on {device}")

    @staticmethod
    def _pool(hidden_state, attention_mask):
        flipped = attention_mask.flip(dims=[1])
        last_pos = flipped.argmax(dim=1)
        col = attention_mask.shape[1] - last_pos - 1
        row = torch.arange(hidden_state.shape[0], device=hidden_state.device)
        return hidden_state[row, col]

    def _finalize(self, embedding):
        truncated = embedding[:self.dimensions]
        norm = torch.linalg.norm(truncated)
        if norm > 0:
            truncated = truncated / norm
        return truncated.cpu().float().tolist()

    def _run(self, conversation):
        self._load()
        text = self._processor.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True,
        )
        images, video_inputs, video_kwargs = process_vision_info(
            conversation, return_video_metadata=True, return_video_kwargs=True,
        )
        if video_inputs is not None:
            videos, video_metadata = zip(*video_inputs)
            videos, video_metadata = list(videos), list(video_metadata)
        else:
            videos, video_metadata = None, None

        inputs = self._processor(
            text=[text], images=images, videos=videos,
            video_metadata=video_metadata, return_tensors="pt",
            padding=True, **video_kwargs,
        )
        inputs = {k: v.to(self._model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = self._model(**inputs)
            emb = self._pool(out.last_hidden_state, inputs["attention_mask"])
            emb = F.normalize(emb, p=2, dim=-1)
        return self._finalize(emb[0])

    def embed_video(self, video, instruction="Represent the video for retrieval."):
        """Embed a video — accepts file path or URL."""
        video_ref = video if video.startswith(("http://", "https://")) else "file://" + os.path.abspath(video)
        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": [{
                "type": "video",
                "video": video_ref,
                "fps": VIDEO_FPS,
                "max_frames": VIDEO_MAX_FRAMES,
            }]},
        ]
        return self._run(conversation)

    def embed_text(self, text, instruction="Represent the user's input."):
        """Embed a text string."""
        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": [{"type": "text", "text": text}]},
        ]
        return self._run(conversation)

    def embed_image(self, image, instruction="Represent the user's input."):
        """Embed an image — accepts file path or URL."""
        image_ref = image if image.startswith(("http://", "https://")) else "file://" + os.path.abspath(image)
        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": [{"type": "image", "image": image_ref}]},
        ]
        return self._run(conversation)

    def embed_query(self, query):
        """Embed a search query."""
        return self.embed_text(query, instruction="Retrieve videos relevant to the query.")


model = Qwen3VLEmbedder()
model._load()

## Embed video natively

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload a video (mp4, mov, etc.)
video_path = list(uploaded.keys())[0]

print(f"Embedding {video_path} natively (fps={VIDEO_FPS}, max_frames={VIDEO_MAX_FRAMES})...")
t0 = time.monotonic()
video_emb = model.embed_video(video_path)
elapsed = time.monotonic() - t0

print(f"✓ Done in {elapsed:.1f}s")
print(f"  Embedding dim: {len(video_emb)}")
print(f"  First 5 values: {video_emb[:5]}")

## Compare: native vs frame extraction

In [ ]:
import cv2
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer

# Load SentenceTransformer for frame extraction comparison
st_model = SentenceTransformer(MODEL_NAME)

# Extract frames manually
cap = cv2.VideoCapture(video_path)
video_fps = cap.get(cv2.CAP_PROP_FPS) or 30
step = max(1, int(video_fps / VIDEO_FPS))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frames = []
for i in range(0, total, step):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, frame = cap.read()
    if not ret:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(Image.fromarray(rgb))
    if len(frames) >= VIDEO_MAX_FRAMES:
        break
cap.release()
print(f"Extracted {len(frames)} frames manually")

# Embed frames and average
frame_embs = st_model.encode(frames, convert_to_numpy=True)
frame_avg_emb = np.mean(frame_embs, axis=0)
frame_avg_emb = frame_avg_emb / np.linalg.norm(frame_avg_emb)
frame_avg_emb = frame_avg_emb[:TARGET_DIMS].tolist()

# Cosine similarity between the two approaches
dot = sum(a * b for a, b in zip(video_emb, frame_avg_emb))
print(f"\nSimilarity between native vs frame-extraction: {dot:.4f}")
print("(1.0 = identical, 0.0 = completely different)")

## Semantic search

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return sum(x * y for x, y in zip(a, b))

# Embed some test queries
queries = [
    "marketing campaign for nike",
    "ocean conservation documentary",
    "enterprise software product launch",
]

print("Query similarity to the video:\n")
for q in queries:
    q_emb = model.embed_query(q)
    score = cosine_similarity(q_emb, video_emb)
    print(f"  [{score:.4f}] {q}")